In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
!pip install transformers datasets torchaudio jiwer -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 38.2 MB/s eta 0:00:0000:0100:01


In [4]:
import os
for root, dirs, files in os.walk(f"/kaggle/input/datasets"):
    level = root.replace(f"/kaggle/input/datasets", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 3:  # only go 3 levels deep
        for f in files[:3]:  # show first 3 files per folder
            print(f"{indent}  {f}")

datasets/
  madrishi/
    dysarthric-speech-dataset/
      uaspeech/
        transcripts/
        test/
          F03/
          M05/
        train/
          F02/
          M04/
          F04/
          M01/


In [5]:
BASE_DIR = "/kaggle/input/datasets/madrishi/dysarthric-speech-dataset/uaspeech/"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR  = os.path.join(BASE_DIR, "test")

def collect_files(directory):
    files = []
    for root, dirs, filenames in os.walk(directory):
        for f in filenames:
            if f.endswith(".wav"):
                files.append(os.path.join(root, f))
    return files

train_files = collect_files(TRAIN_DIR)
test_files  = collect_files(TEST_DIR)

print(f"Train files: {len(train_files)}")
print(f"Test files:  {len(test_files)}")
print(f"Example:     {train_files[0]}")

Train files: 17236
Test files:  10539
Example:     /kaggle/input/datasets/madrishi/dysarthric-speech-dataset/uaspeech/train/F02/F02_B1_UW54_M3.wav


In [6]:
TRANSCRIPT_DIR = os.path.join(BASE_DIR, "transcripts")

In [7]:
transcript_files = os.listdir(TRANSCRIPT_DIR)
with open(os.path.join(TRANSCRIPT_DIR, transcript_files[0]), "r") as f:
    print(f.read()[:500])

DISPOSSESS



In [8]:
# Check if transcript filename matches word code
print(transcript_files[:10])  # see the filenames

# Check a few mappings
for f in transcript_files[:5]:
    with open(os.path.join(TRANSCRIPT_DIR, f), "r") as file:
        content = file.read().strip()
    print(f"{f} → {content}")

['F03_B2_UW77_M6.txt', 'F02_B1_CW92_M5.txt', 'F02_B1_CW68_M5.txt', 'M05_B1_CW91_M8.txt', 'F03_B1_CW24_M7.txt', 'F02_B2_UW74_M7.txt', 'M05_B1_UW60_M6.txt', 'F04_B3_CW25_M8.txt', 'M05_B1_CW14_M7.txt', 'M04_B3_CW69_M6.txt']
F03_B2_UW77_M6.txt → DISPOSSESS
F02_B1_CW92_M5.txt → DAY
F02_B1_CW68_M5.txt → HAS
M05_B1_CW91_M8.txt → DOWN
F03_B1_CW24_M7.txt → HAVE


In [9]:
# Step 1: Build transcript lookup dictionary
transcript_lookup = {}
for f in os.listdir(TRANSCRIPT_DIR):
    if f.endswith(".txt"):
        key = f.replace(".txt", "")
        with open(os.path.join(TRANSCRIPT_DIR, f), "r") as file:
            transcript_lookup[key] = file.read().strip()

print(f"Loaded {len(transcript_lookup)} transcripts")
print("Sample:", list(transcript_lookup.items())[:3])

Loaded 22950 transcripts
Sample: [('F03_B2_UW77_M6', 'DISPOSSESS'), ('F02_B1_CW92_M5', 'DAY'), ('F02_B1_CW68_M5', 'HAS')]


In [10]:
import pandas as pd
def parse_filename(filepath):
    filename = os.path.basename(filepath)
    key = filename.replace(".wav", "")
    speaker = key.split("_")[0]                    # e.g. F02
    word = transcript_lookup.get(key, None)        # look up actual word
    return {"path": filepath, "word": word, "speaker": speaker}

# Build dataframes
train_df = pd.DataFrame([parse_filename(f) for f in train_files])
test_df  = pd.DataFrame([parse_filename(f) for f in test_files])

# Drop any rows where word wasn't found
train_df = train_df.dropna(subset=["word"]).reset_index(drop=True)
test_df  = test_df.dropna(subset=["word"]).reset_index(drop=True)

print(f"Train: {len(train_df)} samples")
print(f"Test:  {len(test_df)} samples")
print(train_df.head())

Train: 14206 samples
Test:  7528 samples
                                                path       word speaker
0  /kaggle/input/datasets/madrishi/dysarthric-spe...  AMETHYSTS     F02
1  /kaggle/input/datasets/madrishi/dysarthric-spe...       KILO     F02
2  /kaggle/input/datasets/madrishi/dysarthric-spe...        SHE     F02
3  /kaggle/input/datasets/madrishi/dysarthric-spe...         TO     F02
4  /kaggle/input/datasets/madrishi/dysarthric-spe...         IT     F02


In [11]:
# Step 4 - Check audio properties
import torchaudio

sample_path = train_df["path"][0]
waveform, sample_rate = torchaudio.load(sample_path)

print(f"Sample rate: {sample_rate}")
print(f"Shape: {waveform.shape}")
print(f"Duration: {waveform.shape[1]/sample_rate:.2f} seconds")
print(f"Word: {train_df['word'][0]}")

Sample rate: 16000
Shape: torch.Size([1, 92034])
Duration: 5.75 seconds
Word: AMETHYSTS


In [12]:
import torch
import torchaudio.transforms as T

TARGET_SR = 16000

def load_and_preprocess(filepath):
    waveform, sr = torchaudio.load(filepath)
    
    # Already mono and 16kHz, but handle edge cases
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    
    if sr != TARGET_SR:
        resampler = T.Resample(orig_freq=sr, new_freq=TARGET_SR)
        waveform = resampler(waveform)
    
    waveform = waveform.squeeze().numpy()
    return waveform

# Test it
sample = load_and_preprocess(train_df["path"][0])
print(f"Preprocessed shape: {sample.shape}")
print(f"Min: {sample.min():.4f}, Max: {sample.max():.4f}")

Preprocessed shape: (92034,)
Min: -1.0000, Max: 0.6854


In [13]:
# Step 6 - Build vocabulary
all_words = train_df["word"].tolist() + test_df["word"].tolist()
all_chars = sorted(set("".join(all_words)))

print(f"Unique characters: {all_chars}")
print(f"Number of characters: {len(all_chars)}")

Unique characters: ['-', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
Number of characters: 27


In [14]:
import json

vocab = {char: idx for idx, char in enumerate(all_chars)}
vocab["[PAD]"] = len(vocab)
vocab["[UNK]"] = len(vocab)
vocab["|"]     = len(vocab)   # word boundary/space

print(f"Vocabulary: {vocab}")
print(f"Vocab size: {len(vocab)}")

with open("/kaggle/working/vocab.json", "w") as f:
    json.dump(vocab, f)

print("Vocab saved!")

Vocabulary: {'-': 0, 'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7, 'H': 8, 'I': 9, 'J': 10, 'K': 11, 'L': 12, 'M': 13, 'N': 14, 'O': 15, 'P': 16, 'Q': 17, 'R': 18, 'S': 19, 'T': 20, 'U': 21, 'V': 22, 'W': 23, 'X': 24, 'Y': 25, 'Z': 26, '[PAD]': 27, '[UNK]': 28, '|': 29}
Vocab size: 30
Vocab saved!


In [15]:
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor

tokenizer = Wav2Vec2CTCTokenizer(
    "/kaggle/working/vocab.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer
)

processor.save_pretrained("/kaggle/working/processor")
print("Processor created and saved!")
print(f"Vocab size: {processor.tokenizer.vocab_size}")

Processor created and saved!
Vocab size: 30


In [16]:
class UASpeechDataset():
    def __init__(self, dataframe, processor):
        self.df = dataframe
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        audio = load_and_preprocess(row["path"])
        
        inputs = self.processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt",
            padding=False
        )
        
        # Fixed: use tokenizer directly instead of as_target_processor
        labels = self.processor.tokenizer(row["word"]).input_ids
        
        return {
            "input_values": inputs.input_values.squeeze(),
            "labels": torch.tensor(labels)
        }

train_dataset = UASpeechDataset(train_df, processor)
test_dataset  = UASpeechDataset(test_df, processor)

sample = train_dataset[0]
print(f"Input shape: {sample['input_values'].shape}")
print(f"Label: {sample['labels']}")

Input shape: torch.Size([92034])
Label: tensor([ 1, 13,  5, 20,  8, 25, 19, 20, 19])


In [17]:
# Step 10 - Data Collator
from dataclasses import dataclass
from typing import Dict, List

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: bool = True

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(
            input_features, padding=self.padding, return_tensors="pt"
        )

        # Fixed: use tokenizer directly
        labels_batch = self.processor.tokenizer.pad(
            label_features, padding=self.padding, return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)
print("Collator ready!")

Collator ready!


In [18]:
# Step 11 - Load HuBERT
from transformers import HubertForCTC

model = HubertForCTC.from_pretrained(
    "facebook/hubert-base-ls960",       # Base — fits in T4
    vocab_size=len(processor.tokenizer),
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    ignore_mismatched_sizes=True
)

for param in model.hubert.feature_extractor.parameters():
    param.requires_grad = False

model.gradient_checkpointing_enable()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

HubertForCTC LOAD REPORT from: facebook/hubert-base-ls960
Key            | Status  | 
---------------+---------+-
lm_head.weight | MISSING | 
lm_head.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable parameters: 90,195,872


In [19]:
model.gradient_checkpointing_enable()

In [20]:
from jiwer import wer

def compute_metrics(pred):
    pred_ids = pred.predictions.argmax(axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    # Fixed: use tokenizer directly
    pred_str  = processor.tokenizer.batch_decode(pred_ids)
    label_str = processor.tokenizer.batch_decode(pred.label_ids, group_tokens=False)

    return {"wer": wer(label_str, pred_str)}

print("Metrics ready!")

Metrics ready!


In [21]:
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir="/kaggle/working/hubert-dysarthric-v4",
    group_by_length=False,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    eval_strategy="epoch",
    num_train_epochs=10,
    fp16=True,
    fp16_full_eval=True,        # added
    learning_rate=1e-4,
    warmup_steps=500,
    weight_decay=0.005,
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    logging_steps=50,
    report_to="none",
    dataloader_num_workers=2,
)

In [22]:
# Initialize lm_head with small weights to stabilize training
import torch.nn as nn

nn.init.normal_(model.lm_head.weight, mean=0.0, std=0.02)
nn.init.zeros_(model.lm_head.bias)

print(f"LM head weights reset to small values")
print(f"Weight std: {model.lm_head.weight.std():.4f}")  # should be ~0.02

LM head weights reset to small values
Weight std: 0.0200


In [23]:
from transformers import HubertForCTC
import torch.nn as nn

model = HubertForCTC.from_pretrained(
    "facebook/hubert-base-ls960",
    ignore_mismatched_sizes=True
)

# Fix vocab size
model.lm_head = nn.Linear(768, 30, bias=True)
nn.init.normal_(model.lm_head.weight, mean=0.0, std=0.02)
nn.init.zeros_(model.lm_head.bias)
model.config.vocab_size = 30

# Keep as float32 — Trainer handles fp16 internally
model = model.to("cuda")

print(f"Model dtype:          {next(model.parameters()).dtype}")  # float32
print(f"LM head out_features: {model.lm_head.out_features}")      # 30
print(f"Vocab size:           {model.config.vocab_size}")         # 30

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

HubertForCTC LOAD REPORT from: facebook/hubert-base-ls960
Key            | Status  | 
---------------+---------+-
lm_head.weight | MISSING | 
lm_head.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model dtype:          torch.float32
LM head out_features: 30
Vocab size:           30


In [24]:
from transformers import HubertForCTC
import torch.nn as nn
import torch

# Load model
model = HubertForCTC.from_pretrained(
    "facebook/hubert-base-ls960",
    ignore_mismatched_sizes=True
)

# Fix vocab size
model.lm_head = nn.Linear(768, 30, bias=True)
nn.init.normal_(model.lm_head.weight, mean=0.0, std=0.02)
nn.init.zeros_(model.lm_head.bias)
model.config.vocab_size = 30

# Explicitly enable gradients for every parameter
for param in model.parameters():
    param.requires_grad = True

# Verify before moving to cuda
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")  # should be ~94 million

# Do a manual forward+backward to confirm gradients flow
dummy_input = torch.randn(1, 16000)
dummy_label = torch.tensor([[1, 5, 12, 12, 15]])  # "HELLO"

outputs = model(dummy_input, labels=dummy_label)
outputs.loss.backward()

# Check lm_head gradient
print(f"lm_head grad norm: {model.lm_head.weight.grad.norm():.4f}")  # must not be None or 0

model = model.to("cuda")

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

HubertForCTC LOAD REPORT from: facebook/hubert-base-ls960
Key            | Status  | 
---------------+---------+-
lm_head.weight | MISSING | 
lm_head.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable parameters: 94,394,782
lm_head grad norm: 303.5317


In [25]:
# Check if gradients are actually being computed
for name, param in model.named_parameters():
    if param.grad is not None:
        print(f"{name}: grad norm = {param.grad.norm():.4f}")
    else:
        print(f"{name}: NO GRADIENT")

hubert.masked_spec_embed: NO GRADIENT
hubert.feature_extractor.conv_layers.0.conv.weight: grad norm = 15.1835
hubert.feature_extractor.conv_layers.0.layer_norm.weight: grad norm = 16.3590
hubert.feature_extractor.conv_layers.0.layer_norm.bias: grad norm = 21.9129
hubert.feature_extractor.conv_layers.1.conv.weight: grad norm = 70.8507
hubert.feature_extractor.conv_layers.2.conv.weight: grad norm = 178.5266
hubert.feature_extractor.conv_layers.3.conv.weight: grad norm = 267.9507
hubert.feature_extractor.conv_layers.4.conv.weight: grad norm = 353.2802
hubert.feature_extractor.conv_layers.5.conv.weight: grad norm = 483.6103
hubert.feature_extractor.conv_layers.6.conv.weight: grad norm = 851.5981
hubert.feature_projection.layer_norm.weight: grad norm = 51.8561
hubert.feature_projection.layer_norm.bias: grad norm = 37.6455
hubert.feature_projection.projection.weight: grad norm = 126.8922
hubert.feature_projection.projection.bias: grad norm = 19.0572
hubert.encoder.pos_conv_embed.conv.bias: g

In [27]:
# Check requires_grad AFTER trainer is initialized but BEFORE training
from transformers import Trainer
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

# Check if Trainer modified requires_grad
trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
print(f"Trainable after Trainer init: {trainable:,}")

# Check lm_head specifically
print(f"lm_head requires_grad: {trainer.model.lm_head.weight.requires_grad}")

# Check optimizer — what parameters is it actually updating?
optimizer, scheduler = trainer.create_optimizer_and_scheduler(num_training_steps=100)
optimizer_params = [id(p) for group in optimizer.param_groups for p in group['params']]
print(f"Optimizer parameter count: {len(optimizer_params)}")

Trainable after Trainer init: 94,394,782
lm_head requires_grad: True


TypeError: cannot unpack non-iterable NoneType object

In [28]:
!pip install librosa -q

import librosa
import numpy as np

def load_and_preprocess(filepath):
    waveform, sr = torchaudio.load(filepath)

    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    if sr != TARGET_SR:
        waveform = T.Resample(orig_freq=sr, new_freq=TARGET_SR)(waveform)

    waveform = waveform.squeeze().numpy()

    # Trim leading and trailing silence
    waveform, _ = librosa.effects.trim(
        waveform,
        top_db=20,          # anything 20dB below peak is silence
        frame_length=512,
        hop_length=128
    )

    return waveform

# Test
sample = load_and_preprocess(train_df["path"][0])
print(f"Duration after trim: {len(sample)/16000:.2f}s")  # should be much shorter

Duration after trim: 2.41s


In [29]:
waveform, _ = librosa.effects.trim(
    waveform,
    top_db=30,          # increased from 20 to 30
    frame_length=512,
    hop_length=128
)

sample = load_and_preprocess(train_df["path"][0])
print(f"Duration after trim: {len(sample)/16000:.2f}s")

# Check a few more samples
for i in range(5):
    s = load_and_preprocess(train_df["path"][i])
    print(f"Sample {i} ({train_df['word'][i]}): {len(s)/16000:.2f}s")

Duration after trim: 2.41s
Sample 0 (AMETHYSTS): 2.41s
Sample 1 (KILO): 1.74s
Sample 2 (SHE): 1.84s
Sample 3 (TO): 0.83s
Sample 4 (IT): 0.51s


In [30]:
# Rebuild datasets with silence-trimmed audio
train_dataset = UASpeechDataset(train_df, processor)
test_dataset  = UASpeechDataset(test_df,  processor)

# Verify one sample
sample = train_dataset[0]
print(f"Input shape: {sample['input_values'].shape}")
print(f"Duration:    {sample['input_values'].shape[0]/16000:.2f}s")

Input shape: torch.Size([38528])
Duration:    2.41s


In [31]:
import librosa

def get_trimmed_length(filepath):
    waveform, sr = torchaudio.load(filepath)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sr != TARGET_SR:
        waveform = T.Resample(orig_freq=sr, new_freq=TARGET_SR)(waveform)
    waveform = waveform.squeeze().numpy()
    waveform, _ = librosa.effects.trim(waveform, top_db=30, frame_length=512, hop_length=128)
    return len(waveform)

# Filter out clips shorter than 0.5 seconds (8000 samples)
MIN_SAMPLES = 8000

print("Filtering train...")
train_df = train_df[train_df["path"].apply(
    lambda p: get_trimmed_length(p) >= MIN_SAMPLES
)].reset_index(drop=True)

print("Filtering test...")
test_df = test_df[test_df["path"].apply(
    lambda p: get_trimmed_length(p) >= MIN_SAMPLES
)].reset_index(drop=True)

print(f"Train after filtering: {len(train_df)}")
print(f"Test after filtering:  {len(test_df)}")

Filtering train...
Filtering test...
Train after filtering: 13091
Test after filtering:  7450


In [32]:
model.config.mask_time_prob = 0.0
model.config.mask_feature_prob = 0.0
print("Masking disabled!")

Masking disabled!


In [34]:
# Increase minimum clip length
MIN_SAMPLES = 16000  # 1 second minimum

print("Filtering train...")
train_df_filtered = train_df[train_df["path"].apply(
    lambda p: get_trimmed_length(p) >= MIN_SAMPLES
)].reset_index(drop=True)

print("Filtering test...")
test_df_filtered = test_df[test_df["path"].apply(
    lambda p: get_trimmed_length(p) >= MIN_SAMPLES
)].reset_index(drop=True)

print(f"Train: {len(train_df_filtered)}")
print(f"Test:  {len(test_df_filtered)}")

Filtering train...
Filtering test...
Train: 9940
Test:  5290


In [35]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

trainer.save_model("/kaggle/working/output/model")
processor.save_pretrained("/kaggle/working/output/processor")
print("Saved!")

Epoch,Training Loss,Validation Loss,Wer
1,197.831211,inf,1.000000
2,159.702070,inf,1.000000
3,138.752588,inf,1.000000
4,102.850879,inf,1.000000
5,86.022598,inf,1.000000
6,71.658789,inf,1.000000
7,84.701045,inf,1.000000
8,86.266318,inf,1.000000
9,76.937163,inf,1.000000
10,85.311553,inf,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved!


In [ ]:
# Reload everything from saved files
from transformers import HubertForCTC, Wav2Vec2Processor

processor = Wav2Vec2Processor.from_pretrained("/kaggle/working/processor")
model = HubertForCTC.from_pretrained("/kaggle/working/hubert-dysarthric-v2")
model = model.to("cuda")

print("Model reloaded!")